In [1]:
pip install shap imbalanced-learn

   ---------------------------------------- 0.0/555.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/555.9 kB ? eta -:--:--
   ------------------ --------------------- 262.1/555.9 kB ? eta -:--:--
   ---------------------------------------- 555.9/555.9 kB 1.4 MB/s eta 0:00:00

   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   ---------------------------------------- 2/2 [shap]

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print("CASA0006 - London Road Safety Data Preprocessing (FIXED)")
print("=" * 70)

# 原始数据路径
RAW_DATA_DIR = Path(r"D:\0.CASA_USS\CASA0006-London-Road-Safety\data\raw")

# 输出路径
OUTPUT_DIR = Path(r"D:\0.CASA_USS\CASA0006-London-Road-Safety\data\processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ========== ONS Borough Code → Name Mapping ==========
# Source: ONS Open Geography Portal, December 2018 boundaries
# These are the official codes used in STATS19 2022-2023

LONDON_BOROUGH_MAPPING = {
    # Inner London (13 boroughs)
    'E09000001': 'City of London',
    'E09000007': 'Camden',
    'E09000010': 'Greenwich',
    'E09000011': 'Hackney',
    'E09000012': 'Hammersmith and Fulham',
    'E09000013': 'Islington',
    'E09000014': 'Kensington and Chelsea',
    'E09000016': 'Lambeth',
    'E09000017': 'Lewisham',
    'E09000022': 'Southwark',
    'E09000028': 'Tower Hamlets',
    'E09000030': 'Wandsworth',
    'E09000033': 'Westminster',
    
    # Outer London (20 boroughs)
    'E09000002': 'Barking and Dagenham',
    'E09000003': 'Barnet',
    'E09000004': 'Bexley',
    'E09000005': 'Brent',
    'E09000006': 'Bromley',
    'E09000008': 'Croydon',
    'E09000009': 'Ealing',
    'E09000015': 'Enfield',
    'E09000018': 'Haringey',
    'E09000019': 'Harrow',
    'E09000020': 'Havering',
    'E09000021': 'Hillingdon',
    'E09000023': 'Hounslow',
    'E09000024': 'Kingston upon Thames',
    'E09000025': 'Merton',
    'E09000026': 'Newham',
    'E09000027': 'Redbridge',
    'E09000029': 'Richmond upon Thames',
    'E09000031': 'Sutton',
    'E09000032': 'Waltham Forest'
}

print(f"\n✓ Loaded mapping for {len(LONDON_BOROUGH_MAPPING)} London boroughs")

# ========== 1. Load and Filter Collision Data ==========
print("\n" + "=" * 70)
print("[1/6] Loading collision data")
print("=" * 70)

collision_files = [
    "dft-road-casualty-statistics-collision-2022.csv",
    "dft-road-casualty-statistics-collision-2023.csv"
]

london_collisions = []

for file in collision_files:
    year = file[-8:-4]
    print(f"\n  Reading: {file}")
    
    df = pd.read_csv(RAW_DATA_DIR / file, low_memory=False)
    
    # Filter for London (police_force 1=Met Police, 48=City of London)
    london_df = df[df['police_force'].isin([1, 48])].copy()
    
    print(f"    → {len(london_df):,} collisions in London {year}")
    
    # Check local_authority_ons_district quality
    if 'local_authority_ons_district' in london_df.columns:
        # Count valid London borough codes
        valid_codes = london_df['local_authority_ons_district'].isin(LONDON_BOROUGH_MAPPING.keys()).sum()
        missing_codes = (london_df['local_authority_ons_district'] == 'E99999999').sum()
        other_codes = len(london_df) - valid_codes - missing_codes
        
        print(f"    → ONS district codes:")
        print(f"       Valid London boroughs: {valid_codes:,} ({100*valid_codes/len(london_df):.1f}%)")
        print(f"       Missing (E99999999): {missing_codes:,} ({100*missing_codes/len(london_df):.1f}%)")
        if other_codes > 0:
            print(f"       Other codes: {other_codes:,} ({100*other_codes/len(london_df):.1f}%)")
        
        # Decode ONS codes to borough names
        london_df['borough_name'] = london_df['local_authority_ons_district'].map(LONDON_BOROUGH_MAPPING)
        
        # For missing/invalid codes, keep as "Unknown"
        london_df['borough_name'].fillna('Unknown', inplace=True)
        
        print(f"    → Decoded to borough names")
        
    else:
        print(f"    ⚠️ 'local_authority_ons_district' column not found!")
        london_df['borough_name'] = 'Unknown'
    
    london_collisions.append(london_df)

# Combine years
collisions = pd.concat(london_collisions, ignore_index=True)

print(f"\n{'=' * 70}")
print(f"Total London collisions (2022-2023): {len(collisions):,}")
print(f"{'=' * 70}")

# ========== 2. Verify Borough Distribution ==========
print("\n[2/6] Verifying borough distribution")

borough_dist = collisions['borough_name'].value_counts()
print(f"\n  Collisions by borough (top 15):")
for i, (borough, count) in enumerate(borough_dist.head(15).items(), 1):
    pct = 100 * count / len(collisions)
    print(f"    {i:2d}. {borough:30s} {count:6,} ({pct:5.2f}%)")

unknown_count = borough_dist.get('Unknown', 0)
if unknown_count > 0:
    print(f"\n  ⚠️ Unknown borough: {unknown_count:,} ({100*unknown_count/len(collisions):.1f}%)")
    print(f"     These will be excluded from spatial stratification")
else:
    print(f"\n  ✓ All collisions successfully mapped to boroughs")

# ========== 3. Create Binary Severity Target ==========
print("\n[3/6] Creating binary severity variable")

# Original: 1=Fatal, 2=Serious, 3=Slight
# Binary: 1 if Fatal/Serious (codes 1-2), 0 if Slight (code 3)
collisions['severity_binary'] = (collisions['collision_severity'].isin([1, 2])).astype(int)

severity_counts = collisions['collision_severity'].value_counts().sort_index()
print(f"\n  Original severity distribution:")
for code, count in severity_counts.items():
    pct = 100 * count / len(collisions)
    severity_label = {1: 'Fatal', 2: 'Serious', 3: 'Slight'}.get(code, 'Unknown')
    print(f"    {severity_label:8s} (code {code}): {count:6,} ({pct:5.2f}%)")

binary_counts = collisions['severity_binary'].value_counts()
print(f"\n  Binary target distribution:")
print(f"    Slight (0):        {binary_counts.get(0, 0):6,} ({100*binary_counts.get(0,0)/len(collisions):5.2f}%)")
print(f"    Serious/Fatal (1): {binary_counts.get(1, 0):6,} ({100*binary_counts.get(1,0)/len(collisions):5.2f}%)")
print(f"    Imbalance ratio:   {binary_counts.get(0,0)/binary_counts.get(1,0):.2f}:1")

# ========== 4. Save Collision Data ==========
print("\n[4/6] Saving collision data")

output_path = OUTPUT_DIR / "london_collisions_2022_2023.csv"
collisions.to_csv(output_path, index=False)

file_size_mb = output_path.stat().st_size / (1024 ** 2)
print(f"\n  ✓ Saved: {output_path}")
print(f"    → {len(collisions):,} rows × {collisions.shape[1]} columns")
print(f"    → File size: {file_size_mb:.1f} MB")

# ========== 5. Load and Save Casualty Data ==========
print("\n[5/6] Loading casualty data")

casualty_files = [
    "dft-road-casualty-statistics-casualty-2022.csv",
    "dft-road-casualty-statistics-casualty-2023.csv"
]

london_casualties = []

for file in casualty_files:
    year = file[-8:-4]
    print(f"\n  Reading: {file}")
    
    df = pd.read_csv(RAW_DATA_DIR / file, low_memory=False)
    
    # Filter for London collision_index only
    london_collision_ids = set(collisions['collision_index'])
    london_cas = df[df['collision_index'].isin(london_collision_ids)].copy()
    
    print(f"    → {len(london_cas):,} casualties linked to London collisions")
    london_casualties.append(london_cas)

casualties = pd.concat(london_casualties, ignore_index=True)
print(f"\n  Total London casualties (2022-2023): {len(casualties):,}")

cas_output = OUTPUT_DIR / "london_casualties_2022_2023.csv"
casualties.to_csv(cas_output, index=False)

cas_size_mb = cas_output.stat().st_size / (1024 ** 2)
print(f"\n  ✓ Saved: {cas_output}")
print(f"    → File size: {cas_size_mb:.1f} MB")

# ========== 6. Load and Save Vehicle Data ==========
print("\n[6/6] Loading vehicle data")

vehicle_files = [
    "dft-road-casualty-statistics-vehicle-2022.csv",
    "dft-road-casualty-statistics-vehicle-2023.csv"
]

london_vehicles = []

for file in vehicle_files:
    year = file[-8:-4]
    print(f"\n  Reading: {file}")
    
    df = pd.read_csv(RAW_DATA_DIR / file, low_memory=False)
    
    london_collision_ids = set(collisions['collision_index'])
    london_veh = df[df['collision_index'].isin(london_collision_ids)].copy()
    
    print(f"    → {len(london_veh):,} vehicles linked to London collisions")
    london_vehicles.append(london_veh)

vehicles = pd.concat(london_vehicles, ignore_index=True)
print(f"\n  Total London vehicles (2022-2023): {len(vehicles):,}")

veh_output = OUTPUT_DIR / "london_vehicles_2022_2023.csv"
vehicles.to_csv(veh_output, index=False)

veh_size_mb = veh_output.stat().st_size / (1024 ** 2)
print(f"\n  ✓ Saved: {veh_output}")
print(f"    → File size: {veh_size_mb:.1f} MB")

# ========== Final Validation ==========
print("\n" + "=" * 70)
print("FINAL VALIDATION")
print("=" * 70)

print(f"\n[TEST 1] Row counts")
print(f"  Collisions:  {len(collisions):,}")
print(f"  Casualties:  {len(casualties):,}")
print(f"  Vehicles:    {len(vehicles):,}")

print(f"\n[TEST 2] Referential integrity")
collision_ids_in_cas = casualties['collision_index'].nunique()
collision_ids_in_veh = vehicles['collision_index'].nunique()
print(f"  Unique collision_index in casualties: {collision_ids_in_cas:,}")
print(f"  Unique collision_index in vehicles:   {collision_ids_in_veh:,}")
print(f"  Expected (collision count):            {len(collisions):,}")

print(f"\n[TEST 3] Borough coverage")
valid_boroughs = (collisions['borough_name'] != 'Unknown').sum()
print(f"  Valid borough assignments: {valid_boroughs:,} / {len(collisions):,} ({100*valid_boroughs/len(collisions):.1f}%)")
print(f"  Unique boroughs: {collisions['borough_name'].nunique() - 1}")  # -1 to exclude 'Unknown'

print(f"\n[TEST 4] File sizes (GitHub limit: 100 MB)")
print(f"  london_collisions_2022_2023.csv:  {file_size_mb:6.1f} MB {'✓' if file_size_mb < 100 else '✗ TOO LARGE'}")
print(f"  london_casualties_2022_2023.csv:  {cas_size_mb:6.1f} MB {'✓' if cas_size_mb < 100 else '✗ TOO LARGE'}")
print(f"  london_vehicles_2022_2023.csv:    {veh_size_mb:6.1f} MB {'✓' if veh_size_mb < 100 else '✗ TOO LARGE'}")

print("\n" + "=" * 70)
print("Preprocessing Complete ✓")
print("=" * 70)
print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"\nNext steps:")
print(f"  1. Upload processed files to GitHub")
print(f"  2. Update notebook Cell 5 to use 'borough_name' column")
print(f"  3. Run notebook end-to-end to verify")
print("=" * 70)

CASA0006 - London Road Safety Data Preprocessing (FIXED)

✓ Loaded mapping for 33 London boroughs

[1/6] Loading collision data

  Reading: dft-road-casualty-statistics-collision-2022.csv
    → 23,502 collisions in London 2022
    → ONS district codes:
       Valid London boroughs: 23,477 (99.9%)
       Missing (E99999999): 0 (0.0%)
       Other codes: 25 (0.1%)
    → Decoded to borough names

  Reading: dft-road-casualty-statistics-collision-2023.csv
    → 22,914 collisions in London 2023
    → ONS district codes:
       Valid London boroughs: 22,895 (99.9%)
       Missing (E99999999): 0 (0.0%)
       Other codes: 19 (0.1%)
    → Decoded to borough names

Total London collisions (2022-2023): 46,416

[2/6] Verifying borough distribution

  Collisions by borough (top 15):
     1. Westminster                     2,516 ( 5.42%)
     2. Southwark                       2,211 ( 4.76%)
     3. Tower Hamlets                   2,052 ( 4.42%)
     4. Wandsworth                      2,038 ( 4.39%